In [ ]:
#file: scripts, nex ijm

In [ ]:
input = "D:/Thesis/final/crops/";
output = "D:/Thesis/final/colored_output/";

// --- CALIBRATION & STYLE ---
pixelSize = 0.06484579; 
unit = "micron";
insetVal = 25; 
// ------------------------

setBatchMode(true); 
processFolder(input, output);
setBatchMode(false);
print("Batch complete! Scale bar is now baked into the C1 and Overview images.");

function processFolder(inputDir, outputDir) {
    if (!File.exists(outputDir)) {
        File.makeDirectory(outputDir);
    }
    
    list = getFileList(inputDir);
    for (i = 0; i < list.length; i++) {
        if (endsWith(list[i], "/")) {
            processFolder(inputDir + list[i], outputDir + list[i]);
        } else if (indexOf(list[i], "_C1") >= 0) { 
            baseName = replace(list[i], "_C1.png", "");
            
            for (c=1; c<=5; c++) {
                currFile = baseName + "_C" + c + ".png";
                if (File.exists(inputDir + currFile)) {
                    open(inputDir + currFile);
                    
                    run("Set Scale...", "distance=1 known=" + pixelSize + " unit=" + unit);
                    run("Enhance Contrast", "saturated=0.35");

                    if (c == 1) { 
                        run("Grays"); 
                        // Adding the scale bar
                        run("Scale Bar...", "width=5 height=12 font=18 color=White background=None location=[Lower Right] bold overlay inset=" + insetVal + " hide");
                        // CRITICAL: This flattens the scale bar into the pixel data
                        run("Flatten"); 
                        // The 'Flatten' command creates a new 'RGB' window, so we close the old one
                        close("\\Default"); 
                    } else {
                        if (c == 2) { run("Red"); } 
                        else if (c == 3) { 
                            reds = newArray(256); greens = newArray(256); blues = newArray(256);
                            for (j=0; j<256; j++) {
                                reds[j] = j; greens[j] = j * 174/255; blues[j] = 0;
                            }
                            setLut(reds, greens, blues);
                        } 
                        else if (c == 4) { run("Green"); } 
                        else if (c == 5) { run("Cyan"); }
                        run("RGB Color");
                    }

                    // Save the individual colored channel (with scale bar if C1)
                    saveAs("PNG", outputDir + currFile);
                    rename("final" + c); 
                }
            }
            
            // --- CREATE THE OVERVIEW ---
            run("Images to Stack", "name=Stack title=final use");
            // This lines up all 5 channels in a row
            run("Make Montage...", "columns=5 rows=1 scale=1 border=0");
            saveAs("PNG", outputDir + baseName + "_Overview.png");
            
            // Cleanup
            close(); 
            if (isOpen("Stack")) {
                selectWindow("Stack");
                close();
            }
        }
    }
}